# Programma 1 - Confronto tra corpora

Confronto linguistico tra due corpora in inglese:

- **Corpus 1** (`Corpora_1.txt`): prosa letteraria — estratto da *Pride and Prejudice* di Jane Austen.
- **Corpus 2** (`Corpora_2.txt`): articoli scientifici — tre articoli di psicologia.

## Setup: librerie, dati e preprocessing

In [12]:
# Import delle librerie e download delle risorse NLTK

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.tag import pos_tag
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords, wordnet
from collections import Counter, defaultdict


nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...

True

In [13]:
# Lettura dei corpora
# I file si trovano nella stessa cartella del notebook,
# quindi si usano percorsi relativi e codifica UTF-8

with open("Corpora_1.txt", "r", encoding="utf-8") as f:
    testo1 = f.read()

with open("Corpora_2.txt", "r", encoding="utf-8") as f:
    testo2 = f.read()

# Verifica: stampa i primi 200 caratteri di ciascun corpus
# per controllare che la lettura sia avvenuta correttamente
print("=== Inizio Corpus 1 ===")
print(testo1[:200])
print("\n=== Inizio Corpus 2 ===")
print(testo2[:200])

=== Inizio Corpus 1 ===
It is a truth universally acknowledged, that a single man in possession of a good fortune must be in want of a wife. However little known the feelings or views of such a man may be on his first enteri

=== Inizio Corpus 2 ===
Introduction Social inequality and injustice are based on group memberships, like gender, race/ethnicity, and class. For example, women and gender minorities experience workplace discrimination, and r


In [14]:
# Preprocessing: sentence splitting, tokenizzazione e PoS tagging

# Suddivisione del testo in frasi
frasi1 = sent_tokenize(testo1)
frasi2 = sent_tokenize(testo2)

# Tokenizzazione di ogni frase: si ottiene una lista di liste di token
token1 = [word_tokenize(frase) for frase in frasi1]
token2 = [word_tokenize(frase) for frase in frasi2]

# Lista piatta di tutti i token (utile per calcoli sull'intero corpus)
token1_flat = [token for frase in token1 for token in frase]
token2_flat = [token for frase in token2 for token in frase]

# PoS tagging: a ogni token viene associata la sua categoria grammaticale
pos1 = [pos_tag(frase) for frase in token1]
pos2 = [pos_tag(frase) for frase in token2]


print(f"Corpus 1 — Frasi: {len(frasi1)}, Token totali: {len(token1_flat)}")
print(f"Corpus 2 — Frasi: {len(frasi2)}, Token totali: {len(token2_flat)}")

Corpus 1 — Frasi: 626, Token totali: 20744
Corpus 2 — Frasi: 825, Token totali: 23884


In [15]:
# Lemmatizzazione

# Funzione che converte i tag PoS di NLTK nel formato richiesto da WordNetLemmatizer

def get_wordnet_pos(tag):
    if tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # categoria di default per i casi non coperti

lemmatizer = WordNetLemmatizer()

# Lemmatizzazione di ogni token usando il suo PoS tag
lemmi1 = [[lemmatizer.lemmatize(token, get_wordnet_pos(tag)) for token, tag in frase] for frase in pos1]
lemmi2 = [[lemmatizer.lemmatize(token, get_wordnet_pos(tag)) for token, tag in frase] for frase in pos2]

## 1-2. Statistiche di base: frasi, token, lunghezze medie

In [16]:
# Funzione che calcola le statistiche di base di un corpus:
# numero di frasi, numero di token, lunghezza media delle frasi (in token)
# e lunghezza media dei token in caratteri (escludendo la punteggiatura)

def statistiche_base(frasi_tok, token_flat):
    num_frasi = len(frasi_tok)
    num_token = len(token_flat)

    # Lunghezza media delle frasi: media del numero di token per ogni frase
    lunghezze_frasi = [len(frase) for frase in frasi_tok]
    lunghezza_media_frasi = sum(lunghezze_frasi) / num_frasi

    # Lunghezza media dei token in caratteri, escludendo la punteggiatura
    token_senza_punteggiatura = [t for t in token_flat if any(c.isalnum() for c in t)]
    lunghezza_media_token = sum(len(t) for t in token_senza_punteggiatura) / len(token_senza_punteggiatura)

    return num_frasi, num_token, lunghezza_media_frasi, lunghezza_media_token

# Calcolo delle statistiche per entrambi i corpora
stat1 = statistiche_base(token1, token1_flat)
stat2 = statistiche_base(token2, token2_flat)

print("=== Corpus 1 ===")
print(f"Numero di frasi: {stat1[0]}")
print(f"Numero di token: {stat1[1]}")
print(f"Lunghezza media delle frasi (in token): {stat1[2]:.2f}")
print(f"Lunghezza media dei token (in caratteri): {stat1[3]:.2f}")

print("\n=== Corpus 2 ===")
print(f"Numero di frasi: {stat2[0]}")
print(f"Numero di token: {stat2[1]}")
print(f"Lunghezza media delle frasi (in token): {stat2[2]:.2f}")
print(f"Lunghezza media dei token (in caratteri): {stat2[3]:.2f}")

=== Corpus 1 ===
Numero di frasi: 626
Numero di token: 20744
Lunghezza media delle frasi (in token): 33.14
Lunghezza media dei token (in caratteri): 4.34

=== Corpus 2 ===
Numero di frasi: 825
Numero di token: 23884
Lunghezza media delle frasi (in token): 28.95
Lunghezza media dei token (in caratteri): 5.69


## 3. Distribuzione delle PoS e confronto tra i corpora

In [17]:
# Versione piatta delle coppie (token, PoS) per analisi sull'intero corpus
pos1_flat = [coppia for frase in pos1 for coppia in frase]
pos2_flat = [coppia for frase in pos2 for coppia in frase]

# Funzione che calcola la distribuzione delle PoS per le prime n coppie
# e per l'intero corpus, usando Counter
def distribuzione_pos(pos_flat):
    finestre = [1000, 2000, 3000]
    distribuzioni = {}
    for n in finestre:
        tags = [tag for token, tag in pos_flat[:n]]
        distribuzioni[f"primi {n} token"] = Counter(tags)
    tags_tutti = [tag for token, tag in pos_flat]
    distribuzioni["intero corpus"] = Counter(tags_tutti)
    return distribuzioni

dist_pos1 = distribuzione_pos(pos1_flat)
dist_pos2 = distribuzione_pos(pos2_flat)

# Confronto diretto tra i due corpora per ogni finestra:
# per ogni categoria grammaticale si mostrano le frequenze affiancate
for finestra in dist_pos1:
    print(f"\n===== Confronto distribuzione PoS - {finestra} =====")
    # Unione delle categorie presenti in almeno uno dei due corpora, ordinate per frequenza nel Corpus 1
    categorie = sorted(
        set(dist_pos1[finestra]) | set(dist_pos2[finestra]),
        key=lambda tag: dist_pos1[finestra].get(tag, 0),
        reverse=True
    )
    print(f"{'PoS':<8}{'Corpus 1':<12}{'Corpus 2':<12}")
    for tag in categorie:
        freq1 = dist_pos1[finestra].get(tag, 0)
        freq2 = dist_pos2[finestra].get(tag, 0)
        print(f"{tag:<8}{freq1:<12}{freq2:<12}")


===== Confronto distribuzione PoS - primi 1000 token =====
PoS     Corpus 1    Corpus 2    
NN      103         198         
PRP     103         8           
IN      100         111         
JJ      78          104         
NNP     73          15          
DT      56          48          
,       54          87          
VB      54          17          
RB      47          35          
VBP     40          34          
.       38          37          
PRP$    32          7           
CC      29          69          
MD      28          6           
NNS     27          105         
VBZ     23          22          
TO      21          16          
VBD     21          0           
VBN     16          26          
CD      12          3           
:       10          2           
WRB     9           6           
VBG     8           22          
WP      6           1           
RBR     3           1           
RP      2           0           
''      2           0           
JJR     2       

## 4. Dimensione del vocabolario e ricchezza lessicale (TTR)

In [18]:
# Funzione che calcola, per porzioni incrementali di 300 token,
# la dimensione del vocabolario (numero di parole distinte)
# e la TTR (Type-Token Ratio = vocabolario / token)

def vocabolario_ttr_incrementale(token_flat):
    risultati = {}
    step = 300
    for n in range(step, len(token_flat) + 1, step):
        porzione = token_flat[:n]
        # Normalizzazione in minuscolo: "The" e "the" sono lo stesso type
        porzione_normalizzata = [t.lower() for t in porzione]
        vocabolario = set(porzione_normalizzata)
        ttr = len(vocabolario) / len(porzione_normalizzata)
        risultati[n] = (len(vocabolario), ttr)

    # Aggiunta del risultato sull'intero corpus, se la lunghezza non è già un multiplo di 300
    n_totale = len(token_flat)
    if n_totale not in risultati:
        porzione_normalizzata = [t.lower() for t in token_flat]
        vocabolario = set(porzione_normalizzata)
        ttr = len(vocabolario) / len(porzione_normalizzata)
        risultati[n_totale] = (len(vocabolario), ttr)

    return risultati

voc_ttr1 = vocabolario_ttr_incrementale(token1_flat)
voc_ttr2 = vocabolario_ttr_incrementale(token2_flat)

print("=== Corpus 1 ===")
for n, (vocab, ttr) in voc_ttr1.items():
    print(f"Primi {n} token -> Vocabolario: {vocab}, TTR: {ttr:.4f}")

print("\n=== Corpus 2 ===")
for n, (vocab, ttr) in voc_ttr2.items():
    print(f"Primi {n} token -> Vocabolario: {vocab}, TTR: {ttr:.4f}")

=== Corpus 1 ===
Primi 300 token -> Vocabolario: 142, TTR: 0.4733
Primi 600 token -> Vocabolario: 230, TTR: 0.3833
Primi 900 token -> Vocabolario: 304, TTR: 0.3378
Primi 1200 token -> Vocabolario: 385, TTR: 0.3208
Primi 1500 token -> Vocabolario: 437, TTR: 0.2913
Primi 1800 token -> Vocabolario: 488, TTR: 0.2711
Primi 2100 token -> Vocabolario: 541, TTR: 0.2576
Primi 2400 token -> Vocabolario: 616, TTR: 0.2567
Primi 2700 token -> Vocabolario: 666, TTR: 0.2467
Primi 3000 token -> Vocabolario: 718, TTR: 0.2393
Primi 3300 token -> Vocabolario: 757, TTR: 0.2294
Primi 3600 token -> Vocabolario: 800, TTR: 0.2222
Primi 3900 token -> Vocabolario: 843, TTR: 0.2162
Primi 4200 token -> Vocabolario: 889, TTR: 0.2117
Primi 4500 token -> Vocabolario: 938, TTR: 0.2084
Primi 4800 token -> Vocabolario: 982, TTR: 0.2046
Primi 5100 token -> Vocabolario: 1028, TTR: 0.2016
Primi 5400 token -> Vocabolario: 1060, TTR: 0.1963
Primi 5700 token -> Vocabolario: 1084, TTR: 0.1902
Primi 6000 token -> Vocabolario: 

## 5. Numero medio di lemmi per frase

In [19]:
# Funzione che calcola il numero medio di lemmi distinti per frase
# Per ogni frase si contano i lemmi distinti (normalizzati in minuscolo),
# poi si fa la media su tutte le frasi del corpus

def media_lemmi_per_frase(lemmi_corpus):
    conteggi = []
    for frase in lemmi_corpus:
        lemmi_normalizzati = [l.lower() for l in frase]
        lemmi_distinti = set(lemmi_normalizzati)
        conteggi.append(len(lemmi_distinti))
    media = sum(conteggi) / len(conteggi)
    return media

media_lemmi1 = media_lemmi_per_frase(lemmi1)
media_lemmi2 = media_lemmi_per_frase(lemmi2)

print(f"Corpus 1 - Numero medio di lemmi distinti per frase: {media_lemmi1:.2f}")
print(f"Corpus 2 - Numero medio di lemmi distinti per frase: {media_lemmi2:.2f}")

Corpus 1 - Numero medio di lemmi distinti per frase: 26.25
Corpus 2 - Numero medio di lemmi distinti per frase: 23.92


## Classificazione di polarità

### Classificazione di polarità: approccio e dati

Per la classificazione di polarità delle frasi si utilizza un classificatore **Multinomial Naive Bayes** con feature **TF-IDF**.

Il classificatore è addestrato sul dataset **movie_reviews**, caricato dalla cartella locale inclusa nel progetto, che contiene 2000 recensioni di film etichettate come positive (`pos`) o negative (`neg`). Il dataset viene diviso in training set (80%) e test set (20%); il parametro `random_state` garantisce la riproducibilità dei risultati.

In [20]:
# Classificatore di polarità basato su Multinomial Naive Bayes con feature TF-IDF,
# addestrato sul dataset movie_reviews (recensioni etichettate come 'pos'/'neg')

from sklearn.datasets import load_files
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

# Caricamento del dataset dalla cartella locale (percorso relativo)
movie_dataset = load_files("movie_reviews", shuffle=True)

# Divisione in training set (80%) e test set (20%)
# random_state fissa il seme casuale per rendere i risultati riproducibili
X_train, X_test, y_train, y_test = train_test_split(
    movie_dataset.data,
    movie_dataset.target,
    test_size=0.20,
    random_state=32
)

# Pipeline che combina l'estrazione delle feature (TF-IDF) e il classificatore
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(tokenizer=word_tokenize, max_features=3000)),
    ('classifier', MultinomialNB())
])

# Addestramento del modello sul training set
modello_polarita = pipeline.fit(X_train, y_train)

# Valutazione delle prestazioni sul test set
predetti = modello_polarita.predict(X_test)
print(classification_report(y_test, predetti, target_names=movie_dataset.target_names))

C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


              precision    recall  f1-score   support

         neg       0.82      0.85      0.84       197
         pos       0.85      0.82      0.83       203

    accuracy                           0.83       400
   macro avg       0.84      0.84      0.83       400
weighted avg       0.84      0.83      0.83       400



### Prestazioni del classificatore

Il classificatore raggiunge un'**accuratezza dell'83%** sul test set, con valori di precision, recall e f1-score bilanciati tra le due classi (circa 0.82–0.85). Le prestazioni sono soddisfacenti considerando la semplicità del modello.

Va però considerato un aspetto importante, ovvero che il classificatore è addestrato su recensioni di film, un dominio diverso da quello dei corpora analizzati. Il modello tende dunque a riconoscere come "positivo" il lessico valutativo tipico delle recensioni, che può non corrispondere al tono effettivo di testi appartenenti ad altri generi.

## 6-7. Distribuzione di polarità e polarità del documento

In [21]:
# Funzione che classifica le frasi di un corpus come positive o negative
# usando il modello addestrato, e calcola la distribuzione di polarità
# e la polarità del documento secondo la formula (P - N) / (P + N)

def analisi_polarita(frasi):
    predizioni = modello_polarita.predict(frasi)

    num_pos = sum(1 for p in predizioni if movie_dataset.target_names[p] == "pos")
    num_neg = sum(1 for p in predizioni if movie_dataset.target_names[p] == "neg")

    totale = num_pos + num_neg
    perc_pos = num_pos / totale * 100
    perc_neg = num_neg / totale * 100

    polarita_documento = (num_pos - num_neg) / (num_pos + num_neg)

    return num_pos, num_neg, perc_pos, perc_neg, polarita_documento

# Le frasi vengono passate come stringhe (non come token): la pipeline TF-IDF tokenizza internamente
pol1 = analisi_polarita(frasi1)
pol2 = analisi_polarita(frasi2)

print("=== Corpus 1 ===")
print(f"Frasi positive: {pol1[0]} ({pol1[2]:.2f}%)")
print(f"Frasi negative: {pol1[1]} ({pol1[3]:.2f}%)")
print(f"Polarità del documento: {pol1[4]:.4f}")

print("\n=== Corpus 2 ===")
print(f"Frasi positive: {pol2[0]} ({pol2[2]:.2f}%)")
print(f"Frasi negative: {pol2[1]} ({pol2[3]:.2f}%)")
print(f"Polarità del documento: {pol2[4]:.4f}")

=== Corpus 1 ===
Frasi positive: 310 (49.52%)
Frasi negative: 316 (50.48%)
Polarità del documento: -0.0096

=== Corpus 2 ===
Frasi positive: 605 (73.33%)
Frasi negative: 220 (26.67%)
Polarità del documento: 0.4667


## Confronto tra i corpora

Dal confronto emergono alcune differenze coerenti con i due generi testuali (prosa letteraria vs articolo scientifico):

- Il Corpus 1 ha frasi più lunghe in media (33.14 token contro 28.95), mentre il Corpus 2 ha token più lunghi in caratteri (5.69 contro 4.34), per via del lessico più tecnico.
- Nel Corpus 1 ci sono molti più pronomi personali e nomi propri, dovuti a dialoghi e personaggi. Nel Corpus 2 prevalgono sostantivi (specie plurali) e aggettivi.
- La TTR diminuisce in entrambi i corpora all'aumentare dei token, come previsto.
- Il numero medio di lemmi per frase è un po' più alto nel Corpus 1 (26.25 contro 23.92), in linea con le frasi più lunghe.
- Sulla polarità, il Corpus 1 risulta quasi neutro, mentre il Corpus 2 viene classificato come più positivo. Questo dato va però preso con cautela, visti i limiti del classificatore già descritti.